### 分词
文本分词是将一段连续的文本内容分割成一个个独立的词汇或字，是自然语言处理中的基本任务。</br>
在Python中，可以使用多种方法进行文本分词，如基于规则的方法、基于统计的方法和基于机器学习的方法等。</br>
以jieba库为例，介绍文本分词的例子。

In [1]:
import jieba

text = '自然语言处理技术是最厉害的技术！'

cuts_generator = jieba.cut(text, cut_all=True)  # 全模式分词
# 全模式： 他/ 说/ ：/ 自然/ 自然语言/ 语言/ 处理/ 技术/ 是/ 最/ 厉害/ 的/ 技术/ ！

cuts_generator = jieba.cut(text, cut_all=False)  # 精确模式分词
# 精确模式： 他/ 说/ ：/ 自然语言/ 处理/ 技术/ 是/ 最/ 厉害/ 的/ 技术/ ！

# 去除停用词、特殊字符
stop_words = ['是', '的']
special_words = [',', ':', '：', '&', '__', '！']

# 使用列表推导式去除停用词
result = [word for word in cuts_generator if word not in stop_words + special_words]

print('result: ', result)
# result: 他/ 自然语言/ 处理/ 技术/ 是/ 最/ 厉害/ 技术/ ！

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/y3/spkvf7mn69j10zpv19hmb9300000gn/T/jieba.cache
Loading model cost 0.387 seconds.
Prefix dict has been built successfully.


result:  ['自然语言', '处理', '技术', '最', '厉害', '技术']


### 预处理

In [3]:
from gensim.models import Word2Vec
import jieba
from tqdm import tqdm
import os

# 加载自定义词典
jieba.load_userdict('./data/jieba_dict.txt')

def word_cut(text):
    cuts_generator = jieba.cut(text, cut_all=False)
    return [word for word in cuts_generator if len(word) > 1]

def del_word(words, del_words):
    return [word for word in words if word not in del_words]

def prepare(text, stop_words, special_words):
    words = word_cut(text)
    words = del_word(words, stop_words)
    words = del_word(words, special_words)
    return words

stop_words = ['说', '的', '了', '是']
special_words = ['，', '。', ':', '：', '&', '__', '【', '】', '(', ')', '......']
    
sents = []

# 读取训练数据
files = os.listdir('nlpdata/教育')
for i, file in tqdm(enumerate(files)):
    with open('nlpdata/教育/' + file, 'r') as fr:
        sentences = fr.readlines()

    for sent in sentences:
        sents.append(prepare(sent.strip(), stop_words, special_words))
    
    if i == 3000:
        break
print(len(sents), sents[0])

FileNotFoundError: [Errno 2] No such file or directory: './data/jieba_dict.txt'

## Word2Vec
是一种用于生成词向量的模型，它可以将词语映射到一个高维空间中，使得语义相近的词语在空间中距离较近。
### Word2Vec有两种训练方法：Skip-gram和CBOW。
1、Skip-gram通过给定一个词，预测它周围的上下文词；</br>
2、CBOW则通过给定一组上下文词，预测中心词。

In [11]:
# 训练Word2Vec模型
# size：词向量的维度，默认值为100。较大的值可以捕捉到更多的词汇信息，但计算量会更大。
# window：窗口大小，表示在训练时考虑的单词最远距离，默认值为5。窗口越大，模型越能捕捉到长距离的上下文信息，但计算量也会增加。
# min_count：最小词频阈值，低于该阈值的单词将不会被纳入模型，默认值为5。可以通过设置较小的值来过滤掉高频词。
# sg：训练算法，默认为0，表示使用CBOW算法；设置为1时，表示使用skip-gram算法

# Skip-gram通过给定一个词，预测它周围的上下文词
model = Word2Vec(sents, vector_size=16, window=5, min_count=2, sg=1)
model.save("w2v_skip_gram.model")

# CBOW则通过给定一组上下文词，预测中心词
model = Word2Vec(sents, vector_size=16, window=5, min_count=2, sg=0)
model.save("w2v_cbow.model")

### 词向量查询和相似度计算

In [12]:
from gensim.models import Word2Vec

# 加载模型
model = Word2Vec.load('w2v_skip_gram.model')

# 打印词表
# print(model.wv.key_to_index)

# 查询词向量 - 用数值表示文本
vector = model.wv['北京大学']
print('北京大学: ', vector, '\n')

# 计算相似度
similarity = model.wv.similarity('北京大学', '清华大学')
print("Similarity between '北京大学' and '清华大学':", similarity, '\n')

# 找到与给定单词最相似的单词
similar_words = model.wv.most_similar('北京大学', topn=3)
print(similar_words, '\n')

# 打印词表
# print(model.wv.key_to_index.keys())

北京大学:  [-0.40485036  0.59517956  0.577179    1.125408    0.7334087  -1.1070582
  1.3573309  -1.0495346  -0.8514622   0.23937452 -0.07466317  0.12792102
 -0.18835205 -1.3525242  -0.10592687  0.47149906] 

Similarity between '北京大学' and '清华大学': 0.9613947 

[('清华大学', 0.961394727230072), ('上海交通大学', 0.9385278224945068), ('华中科技大学', 0.9370906352996826)] 

